# PFS x DESI Multi-Tracer Fisher Forecast --- Figures

This notebook reproduces all figures from the paper.

**Three analysis scenarios:**
- **Broad**: conservative Gaussian priors ([Chudaykin+ 2025](https://arxiv.org/abs/2507.13433))
- **Cross-cal**: priors calibrated from the PFS--DESI overlap ($k_{\rm max} = 0.20\,h{\rm Mpc}^{-1}$)
- **Fixed nuisance**: nuisance parameters perfectly known (theoretical ceiling)

**SBP benchmark lines:**
- **SBP, PS**: power-spectrum-level simulation-based prior ([Zhang+ 2025](https://arxiv.org/abs/2409.12937))
- **SBP, FL**: field-level simulation-based prior ([Chudaykin+ 2026](https://arxiv.org/abs/2602.18554))

**Key parameters:**
- `f_shared = 0.045`: shared ELG fraction between PFS and DESI, `f_shared=n_shared/n_PFS`
- `r_sigma_v = 0.75`: FoG velocity ratio `sigma_v,PFS / sigma_v,DESI`
- Asymmetric kmax in overlap: `kmax_PFS = kmax_DESI / r_sigma_v`

## Updated headline analysis: joint multi-tracer Fisher

The headline forecast in the paper is now produced by
`scripts/run_joint_fisher.py`, which builds a single joint multi-tracer
Fisher across the full DESI footprint with the PFS overlap contributing
in the 1,200 deg² overlap (volume-partitioned per
`pfsfog/fisher_joint.py`). This replaces the legacy two-stage pipeline
below (Step 1 calibration → Step 2 cosmology with priors), which
double-counts DESI auto-spectra. The legacy cells are retained for
diagnostic purposes (calibration efficiency per z-bin, r_sigma_v
sensitivity).

In [ ]:
# CPU/thread setup — must run before importing numpy / jax / scipy / mkl-backed libs.
import os
from pathlib import Path

if Path().resolve().name == "notebooks":
    os.chdir(Path().resolve().parent)  # -> repo root

# Use all logical CPUs for BLAS / OpenMP / numexpr / Apple Accelerate.
n_cpus = str(os.cpu_count() or 1)
for var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",   # macOS Accelerate
    "BLIS_NUM_THREADS",
):
    os.environ.setdefault(var, n_cpus)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# JAX/XLA on CPU: Eigen/OpenMP-driven threading respects OMP_NUM_THREADS;
# enable the multi-threaded Eigen backend explicitly for safety.
os.environ.setdefault(
    "XLA_FLAGS",
    "--xla_cpu_multi_thread_eigen=true --xla_cpu_enable_fast_math=true"
)

import sys
sys.path.insert(0, ".")

import jax
jax.config.update("jax_enable_x64", True)
print(f"CPUs: {n_cpus}, JAX devices: {jax.devices()}")

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams.update({
    "text.usetex": False,
    "font.size": 24})
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 6.0
plt.rcParams['ytick.major.size'] = 6.0
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['xtick.minor.size'] = 4.0
plt.rcParams['ytick.minor.size'] = 4.0
plt.rcParams['xtick.minor.width'] = 0.8
plt.rcParams['ytick.minor.width'] = 0.8
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['font.size'] = 24
plt.rcParams['axes.linewidth'] = 1.5


In [ ]:
# Run the pipeline (~30s)
from pfsfog.config import ForecastConfig
from pfsfog.cli import run_pipeline

cfg = ForecastConfig.from_yaml("configs/default.yaml")
results = run_pipeline(cfg, verbose=True)

In [ ]:
from pfsfog.plots import set_style
set_style()

# Convenient references
overlap = results.overlap_results        # {(zlo,zhi): OverlapResult}
scenarios = results.scenario_results      # {"broad": ScenarioResult, ...}
z_bins = results.config.z_bins
sorted_zbins = sorted(overlap.keys())

from pfsfog.eft_params import broad_priors, NUISANCE_NAMES, HOD_BENCHMARK, FIELD_LEVEL_BENCHMARK
from pfsfog.scenarios import SCENARIOS, compute_calibration_efficiency

COLORS = {"broad": "C0", "cross-cal": "C2", "oracle": "C1"}
LABELS = {"broad": "Broad", "cross-cal": "Calibrated", "oracle": "Fixed"}
scenario_names = [s.name for s in SCENARIOS]
cosmo_params = ["fsigma8", "Mnu", "Omegam"]

## Fig. 1 — Per-(tracer, z-bin) nuisance constraints (diagnostic)

Compares the joint-Fisher per-z-bin nuisance constraints, projected
onto each DESI~DR2 sample, with the conservative broad priors.
Diagnostic of where the multi-tracer information most efficiently
tightens nuisance parameters.

In [ ]:
# Fig. 1 (compute) — extract per-(tracer, z-bin) nuisance σ from the joint Fisher.
from pfsfog.config import ForecastConfig
from pfsfog.cosmo import FiducialCosmology
from pfsfog.eft_params import broad_priors, NUISANCE_NAMES
from pfsfog.fisher_joint import run_joint_fisher
from pfsfog.surveys import SurveyGroup, desi_elg, desi_lrg, desi_qso, pfs_elg
from scripts.run_desi_multisample import DR2_SAMPLES
from scripts.run_joint_fisher import ZBINS
from ps_1loop_jax import PowerSpectrum1Loop

cfg_n = ForecastConfig.from_yaml('configs/default.yaml')
cosmo_n = FiducialCosmology(backend=cfg_n.cosmo_backend)
ps_n = PowerSpectrum1Loop(do_irres=False)
sg_n = SurveyGroup(
    pfs=pfs_elg(),
    desi_tracers={'DESI-ELG': desi_elg(), 'DESI-LRG': desi_lrg(), 'DESI-QSO': desi_qso()},
    overlap_area_deg2=cfg_n.overlap_area_deg2,
    desi_full_area_deg2=cfg_n.desi_area_deg2,
    pfs_zmax=1.6,
)
# `parallel=True` opts into multi-process per-z-bin Fisher (≈1.9× speedup
# with warm JAX cache; cache lives in .cache/jax). Set parallel=False to
# run sequentially (numerically identical, just slower).
PARALLEL = True
N_WORKERS = None  # auto: min(n_zbins, cpu_count, 8)

FIG1_RES = run_joint_fisher(cfg_n, cosmo_n, ps_n, sg_n, include_pfs=True, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)
FIG1_FINV = np.linalg.inv(FIG1_RES.fisher.F)
FIG1_PN = FIG1_RES.fisher.param_names

# Per (DR2 sample, nuisance parameter), average σ across the sample's overlap z-bins.
FIG1_BP = broad_priors().sigma_dict()
FIG1_SAMPLE_CAL = {}
for sample in DR2_SAMPLES:
    avg = {}
    if not sample.overlap_zbins:
        FIG1_SAMPLE_CAL[sample.name] = {n: FIG1_BP[n] for n in NUISANCE_NAMES}
        continue
    for n in NUISANCE_NAMES:
        sigmas = []
        for zlo, zhi in sample.overlap_zbins:
            key = f'{n}_{sample.tracer}_z{zlo:.2f}-{zhi:.2f}'
            if key in FIG1_PN:
                idx = FIG1_PN.index(key)
                sigmas.append(np.sqrt(FIG1_FINV[idx, idx]))
        avg[n] = float(np.mean(sigmas)) if sigmas else (FIG1_BP[n] if FIG1_BP[n] is not None else np.nan)
    FIG1_SAMPLE_CAL[sample.name] = avg
print('FIG1_SAMPLE_CAL ready.')

In [ ]:
# Fig. 1 (plot) — bar (broad) + markers (per-sample σ from joint Fisher),
# with horizontal offset between samples for legibility.
params_to_show = list(NUISANCE_NAMES)
plabels = {
    'b1_sigma8': r'$b_1\sigma_8$',
    'b2_sigma8sq': r'$b_2\sigma_8^2$',
    'bG2_sigma8sq': r'$b_{\mathcal{G}_2}\sigma_8^2$',
    'bGamma3': r'$b_{\Gamma_3}$',
    'c0': r'$c_0$', 'c2': r'$c_2$', 'c4': r'$c_4$',
    'c_tilde': r'$\tilde{c}$', 'c1': r'$c_1$',
    'Pshot': r'$P_{\rm shot}$', 'a0': r'$a_0$', 'a2': r'$a_2$',
}
x = np.arange(len(params_to_show))
sample_colors = {
    'LRG1': '#BBBBBB', 'LRG2': '#CC6677', 'LRG3': '#882255',
    'ELG1': '#117733', 'ELG2': '#44AA99', 'QSO': '#332288',
}
sample_markers = {'LRG1': 'v', 'LRG2': 'v', 'LRG3': 'v',
                  'ELG1': 'o', 'ELG2': 'o', 'QSO': 's'}
samples_in_order = list(FIG1_SAMPLE_CAL.keys())
n_samples = len(samples_in_order)

fig, ax = plt.subplots(figsize=(14, 6))
# Broad-prior bars
broad_vals = [FIG1_BP[n] if FIG1_BP[n] is not None else np.nan for n in params_to_show]
ax.bar(x, broad_vals, width=0.7, color='lightgray', alpha=0.5, label='Broad prior')
# Per-sample markers with horizontal offset
for isamp, sample_name in enumerate(samples_in_order):
    avg = FIG1_SAMPLE_CAL[sample_name]
    yvals = [avg.get(n, np.nan) for n in params_to_show]
    offset = (isamp - n_samples / 2 + 0.5) * 0.10
    ax.scatter(x + offset, yvals,
               marker=sample_markers[sample_name],
               s=110, color=sample_colors[sample_name],
               edgecolor='k', linewidth=0.5,
               label=sample_name, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels([plabels[p] for p in params_to_show])
ax.set_ylabel(r'$\sigma$ (joint Fisher, marginalized)')
ax.set_yscale('log')
ax.legend(frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.15))
fig.tight_layout()
fig.savefig('paper/figs/calibrated_vs_broad.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 2 — Joint Fisher headline (DESI-only vs DESI+PFS)

Marginalized 1σ uncertainties on fσ8, Mν, Ωm from the joint Fisher
across 0.4 < z < 2.1, with the volume-partitioned
F = F_overlap(1,200 deg²) + F_nonoverlap(12,800 deg²).

In [ ]:
# Fig. 2 (compute) — run the three joint-Fisher scenarios.
import jax
jax.config.update('jax_enable_x64', True)
from pfsfog.config import ForecastConfig
from pfsfog.cosmo import FiducialCosmology
from pfsfog.fisher_joint import run_joint_fisher, run_broad_baseline
from pfsfog.surveys import SurveyGroup, desi_elg, desi_lrg, desi_qso, pfs_elg
from pfsfog.eft_params import COSMO_NAMES, HOD_BENCHMARK, FIELD_LEVEL_BENCHMARK
from scripts.run_joint_fisher import ZBINS
from ps_1loop_jax import PowerSpectrum1Loop

cfg_j = ForecastConfig.from_yaml('configs/default.yaml')
cosmo_j = FiducialCosmology(backend=cfg_j.cosmo_backend)
ps_j = PowerSpectrum1Loop(do_irres=False)
sg_j = SurveyGroup(
    pfs=pfs_elg(),
    desi_tracers={'DESI-ELG': desi_elg(), 'DESI-LRG': desi_lrg(), 'DESI-QSO': desi_qso()},
    overlap_area_deg2=cfg_j.overlap_area_deg2,
    desi_full_area_deg2=cfg_j.desi_area_deg2,
    pfs_zmax=1.6,
)
# `parallel=True` opts into multi-process per-z-bin Fisher (≈1.9× speedup
# with warm JAX cache; cache lives in .cache/jax). Set parallel=False to
# run sequentially (numerically identical, just slower).
PARALLEL = True
N_WORKERS = None  # auto: min(n_zbins, cpu_count, 8)

RES_BROAD = run_broad_baseline(cfg_j, cosmo_j, ps_j, sg_j, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)
RES_DESI_ONLY = run_joint_fisher(cfg_j, cosmo_j, ps_j, sg_j, include_pfs=False, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)
RES_JOINT = run_joint_fisher(cfg_j, cosmo_j, ps_j, sg_j, include_pfs=True, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)
print('Fishers ready: RES_BROAD, RES_DESI_ONLY, RES_JOINT')

In [ ]:
# Fig. 2 (plot) — bar chart with SBP benchmark lines.
param_titles = [r'$f\sigma_8$', r'$M_\nu$ [eV]', r'$\Omega_m$']
param_ylabels = [r'$\sigma(f\sigma_8)$', r'$\sigma(M_\nu)$ [eV]', r'$\sigma(\Omega_m)$']
scenario_labels = ['DESI broad\n(single-tracer)', 'DESI-only\njoint', 'DESI+PFS\njoint']
scenario_colors = ['#C44E52', '#4C72B0', '#55A868']
scenario_results = [RES_BROAD, RES_DESI_ONLY, RES_JOINT]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ip, (cp, title, ylabel) in enumerate(zip(COSMO_NAMES, param_titles, param_ylabels)):
    ax = axes[ip]
    x = np.arange(3)
    vals = [r.sigma[cp] for r in scenario_results]
    ax.bar(x, vals, color=scenario_colors, width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(scenario_labels, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=16)
    ax.set_title(title, fontsize=18)
    sigma_broad = RES_BROAD.sigma[cp]
    if cp == 'fsigma8':
        hod_imp = HOD_BENCHMARK['sigma8_improvement']
        ax.axhline(sigma_broad * (1 - hod_imp), ls='--', color='gray', lw=1.2,
                   label=fr'SBP, PS ($-{hod_imp*100:.0f}$%)')
        fl_imp = FIELD_LEVEL_BENCHMARK['sigma8_improvement']
        ax.axhline(sigma_broad * (1 - fl_imp), ls=':', color='gray', lw=1.2,
                   label=fr'SBP, FL ($-{fl_imp*100:.0f}$%)')
        ax.legend(frameon=False, loc='upper right', fontsize=12)
    elif cp == 'Omegam':
        hod_imp = HOD_BENCHMARK['Omegam_improvement']
        ax.axhline(sigma_broad * (1 - hod_imp), ls='--', color='gray', lw=1.2,
                   label=fr'SBP, PS ($-{hod_imp*100:.0f}$%)')
        fl_imp = FIELD_LEVEL_BENCHMARK['Omegam_improvement']
        ax.axhline(sigma_broad * (1 - fl_imp), ls=':', color='gray', lw=1.2,
                   label=fr'SBP, FL ($-{fl_imp*100:.0f}$%)')
        ax.legend(frameon=False, loc='upper right', fontsize=12)
fig.tight_layout()
fig.savefig('paper/figs/full_area_constraints.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nHeadline σ values:')
for cp, label in zip(COSMO_NAMES, param_titles):
    sb = RES_BROAD.sigma[cp]; sd = RES_DESI_ONLY.sigma[cp]; sj = RES_JOINT.sigma[cp]
    print(f'  {label:14s}: broad={sb:.4e}  DESI-only={sd:.4e} ({100*(sb-sd)/sb:+.1f}%)  '
          f'DESI+PFS={sj:.4e} ({100*(sb-sj)/sb:+.1f}% vs broad; {100*(sd-sj)/sd:+.1f}% vs DESI-only)')

## Figure 3 — Calibration efficiency per z-bin (diagnostic)

Per-redshift diagnostic of how much of the broad-to-fixed-nuisance
gap is closed by the multi-tracer-calibrated priors. The headline
joint-Fisher constraints don't factorize per z-bin (cosmology is shared);
this figure shows where the multi-tracer information is most effective.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

markers = {"fsigma8": "o", "Mnu": "s", "Omegam": "^"}
clabels = {"fsigma8": r"$f\sigma_8$", "Mnu": r"$M_\nu$", "Omegam": r"$\Omega_m$"}
z_mids = [0.5 * (zb[0] + zb[1]) for zb in z_bins]

for cp in cosmo_params:
    effs = []
    for zb in z_bins:
        sb = scenarios["broad"].sigmas_per_z[zb][cp]
        so = scenarios["oracle"].sigmas_per_z[zb][cp]
        sx = scenarios["cross-cal"].sigmas_per_z[zb][cp]
        eff = compute_calibration_efficiency(sx, sb, so)
        effs.append(eff if eff is not None else 0.0)
    ax.plot(z_mids, effs, marker=markers[cp], label=clabels[cp], lw=1.5)

ax.set_xlabel(r"$z_{\rm eff}$")
ax.set_ylabel("Calibration efficiency")
ax.set_ylim(-0.05, 1.05)
ax.axhline(1.0, ls=":", color="gray", lw=0.5)
ax.axhline(0.0, ls=":", color="gray", lw=0.5)
ax.legend(frameon=False, fontsize=16)
fig.tight_layout()
fig.savefig("paper/figs/calibration_efficiency.png", dpi=200, bbox_inches="tight")
plt.show()

## Figure 4 — Sensitivity to r_sigma_v (diagnostic)

Legacy single-tracer DESI-ELG sensitivity test: the green curve
(asymmetric kmax, kmax^PFS = kmax^DESI / r_sigma_v) shows that the
FoG-velocity-ratio dependence comes entirely from extending the PFS
auto- and PFS×DESI cross-spectra to higher k. The flat red curve
(symmetric kmax) shows that counterterm rescaling alone has no impact.
The headline joint-Fisher constraints inherit this stability.

In [ ]:
r_values = [0.5, 0.6, 0.75, 0.9, 1.0]
sigma_asym = {}
sigma_sym = {}
broad_baseline = None

for r in r_values:
    # Asymmetric kmax (default)
    cfg_r = ForecastConfig(r_sigma_v=r, output_dir="results/_sweep_rsv")
    res_r = run_pipeline(cfg_r, verbose=False)
    sigma_asym[r] = res_r.scenario_results["cross-cal"].sigmas_combined["fsigma8"]
    if broad_baseline is None:
        broad_baseline = res_r.scenario_results["broad"].sigmas_combined["fsigma8"]

    # Symmetric kmax (fixed kmax_PFS = kmax_DESI)
    cfg_s = ForecastConfig(r_sigma_v=r, output_dir="results/_sweep_rsv",
                           kmax_pfs_overlap=0.20, kmax_cross_overlap=0.20)
    res_s = run_pipeline(cfg_s, verbose=False)
    sigma_sym[r] = res_s.scenario_results["cross-cal"].sigmas_combined["fsigma8"]

    imp_a = (broad_baseline - sigma_asym[r]) / broad_baseline * 100
    imp_s = (broad_baseline - sigma_sym[r]) / broad_baseline * 100
    print(f"  r_sv={r:.2f}: asym={sigma_asym[r]:.4e} ({imp_a:+.1f}%)  "
          f"sym={sigma_sym[r]:.4e} ({imp_s:+.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

r_arr = sorted(sigma_asym.keys())
h1, = ax.plot(r_arr, [sigma_asym[r] for r in r_arr], "o-", color="C1",
              lw=2, ms=7, label=r"Asymmetric $k_{\max}$")
h2, = ax.plot(r_arr, [sigma_sym[r] for r in r_arr], "s--", color="C2",
              lw=2, ms=7, label=r"Symmetric $k_{\max}$")

h3 = ax.axhline(broad_baseline, ls=":", color="C0", lw=1, label="Broad baseline")
c = ['silver', 'gray']
h4 = ax.axhline(broad_baseline * 0.9, ls=':', color=c[0], lw=0.8, label=r"$-10$%")
h5 = ax.axhline(broad_baseline * 0.8, ls=':', color=c[1], lw=0.8, label=r"$-20$%")

ax.set_xlabel(r"$r_{\sigma_v} = \sigma_{v,\mathrm{PFS}} / \sigma_{v,\mathrm{DESI}}$")
ax.set_ylabel(r"$\sigma(f\sigma_8)$ combined")

leg1 = ax.legend(
    [h1, h2],
    [r"Asymmetric $k_{\max}$", r"Symmetric $k_{\max}$"],
    frameon=False,
    loc="upper left",
    fontsize=16,
    bbox_to_anchor=(0., 1.00),
)

ax.add_artist(leg1)

leg2 = ax.legend(
    [h3, h4, h5],
    ["Broad baseline", r"$-10$%", r"$-20$%"],
    frameon=False,
    loc="upper right",
    fontsize=16,
    bbox_to_anchor=(1.00, 1.00),
)

fig.tight_layout()
fig.savefig("paper/figs/sensitivity_rsigmav.png", dpi=200, bbox_inches="tight")
plt.show()

## Summary table

In [ ]:
print(f"{'Scenario':<18s} {'kmax':>5s}  {'sigma(fs8)':>10s}  {'D%':>6s}  "
      f"{'sigma(Mnu)':>10s}  {'D%':>6s}  {'sigma(Om)':>10s}  {'D%':>6s}")
print("-" * 85)

sb = {cp: scenarios["broad"].sigmas_combined[cp] for cp in cosmo_params}

for sn in scenario_names:
    sc = [s for s in SCENARIOS if s.name == sn][0]
    row = f"{LABELS[sn]:<18s} {sc.kmax:5.2f}"
    for cp in cosmo_params:
        s = scenarios[sn].sigmas_combined[cp]
        imp = (sb[cp] - s) / sb[cp] * 100
        row += f"  {s:10.4e}  {imp:+5.1f}%"
    print(row)

print()
print(f"kmax_DESI = {cfg.kmax_desi_overlap:.3f}, "
      f"kmax_PFS = {cfg.compute_kmax_pfs():.3f} h/Mpc  "
      f"(r_sv = {cfg.r_sigma_v})")
print(f"f_shared = {cfg.f_shared_elg}")
print(f"Overlap: {cfg.overlap_area_deg2:.0f} deg^2, "
      f"Full DESI: {cfg.desi_area_deg2:.0f} deg^2")
print(f"z-bins: {z_bins}")

## Appendix Figure A1 --- Derivative validation (autodiff vs finite difference)

Left: normalized derivatives |dP_0/dtheta_i| / P_0 for all 15 parameters.
Right: fractional autodiff-vs-finite-difference agreement for all 15 parameters.
Finite differences use adaptive step selection via `numdifftools` (Richardson extrapolation).

In [ ]:
import jax.numpy as jnp
from ps_1loop_jax import PowerSpectrum1Loop
from pfsfog.cosmo import FiducialCosmology, make_plin_func, make_growth_rate_func
from pfsfog.eft_params import desi_elg_fiducials, COSMO_NAMES
from pfsfog.ps1loop_adapter import fisher_to_ps1loop_auto
from pfsfog.derivatives import (
    dPell_dtheta_stencil, dPell_dtheta_autodiff_all_jit,
    dPell_d_fsigma8, dPell_d_cosmo_stencil, dPell_d_cosmo_autodiff,
    _make_mutable,
)

ps = PowerSpectrum1Loop(do_irres=False)
cosmo_d = FiducialCosmology(backend="cosmopower")
z_eff = 0.9
s8 = cosmo_d.sigma8(z_eff); f_z = float(cosmo_d.f(z_eff)); h_d = cosmo_d.params["h"]
pk_data = cosmo_d.pk_data(z_eff)
b1_d, nbar_d = 1.3, 4e-4
fid = desi_elg_fiducials(b1_d, s8)
params_d = fisher_to_ps1loop_auto(fid, s8, f_z, h_d, nbar_d)
k_deriv = jnp.arange(0.01, 0.26, 0.005)
k_np = np.asarray(k_deriv)
P0_fid = np.asarray(ps.get_pk_ell(k_deriv, 0, pk_data, params_d))

# --- Cosmological derivatives: autodiff + finite-diff ---
pkdata_fn = make_plin_func("cosmopower")
f_fn_d = make_growth_rate_func()
cosmo_dict = dict(cosmo_d.params)

# fσ₈: autodiff (via chain rule through f) + finite-diff for validation
cosmo_derivs_d = {
    "fsigma8": np.asarray(dPell_d_fsigma8(ps, k_deriv, pk_data, params_d, s8, 0)),
}
# Finite-diff fσ₈ for right panel comparison
fid_f = params_d["f"]
h_step_f = 0.005
def _pk_of_fs8(delta):
    p = _make_mutable(params_d)
    p["f"] = fid_f + delta * s8
    return np.asarray(ps.get_pk_ell(k_deriv, 0, pk_data, p))
fsigma8_fd = (
    -_pk_of_fs8(2*h_step_f) + 8*_pk_of_fs8(h_step_f)
    - 8*_pk_of_fs8(-h_step_f) + _pk_of_fs8(-2*h_step_f)
) / (12 * h_step_f)

cosmo_ad = {"fsigma8": cosmo_derivs_d["fsigma8"]}
cosmo_fd_dict = {"fsigma8": fsigma8_fd}

# Mν, Ωm: autodiff through cosmopower-jax + finite-diff
for cp in ("Omegam", "Mnu"):
    cosmo_ad[cp] = np.asarray(dPell_d_cosmo_autodiff(
        ps, k_deriv, pkdata_fn, f_fn_d, cosmo_dict, params_d, cp, z_eff, s8, 0))
    cosmo_fd_dict[cp] = np.asarray(dPell_d_cosmo_stencil(
        ps, k_deriv, cosmo_d, params_d, cp, z_eff, s8, 0))
    cosmo_derivs_d[cp] = cosmo_ad[cp]

# --- Nuisance: autodiff (vectorized JIT) + finite-diff (stencil per param) ---
nuis_arr = dPell_dtheta_autodiff_all_jit(ps, k_deriv, pk_data, params_d, s8, ells=(0,))  # (N_nuis, 1, Nk)
nuis_ad = {pn: np.asarray(nuis_arr[ip, 0]) for ip, pn in enumerate(NUISANCE_NAMES)}
nuis_fd = {pn: np.asarray(dPell_dtheta_stencil(ps, k_deriv, pk_data, params_d, pn, s8, 0))
           for pn in NUISANCE_NAMES}

# Plot groups
GROUPS = {
    "Cosmo": (["fsigma8","Mnu","Omegam"],
              [r"$f\sigma_8$",r"$M_\nu$",r"$\Omega_m$"],
              ["#E24A33","#348ABD","#988ED5"]),
    "Bias": (["b1_sigma8","b2_sigma8sq","bG2_sigma8sq","bGamma3"],
             [r"$b_1\sigma_8$",r"$b_2\sigma_8^2$",r"$b_{\mathcal{G}_2}\sigma_8^2$",r"$b_{\Gamma_3}$"],
             ["#2ca02c","#98df8a","#006400","#90EE90"]),
    "Ctr": (["c0","c2","c4","c_tilde","c1"],
            [r"$c_0$",r"$c_2$",r"$c_4$",r"$\tilde{c}$",r"$c_1$"],
            ["#ff7f0e","#ffbb78","#d62728","#ff9896","#bcbd22"]),
    "Stoch": (["Pshot","a0","a2"],
              [r"$P_{\rm shot}$",r"$a_0$",r"$a_2$"],
              ["#17becf","#9edae5","#7f7f7f"]),
}

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# -----------------------------
# LEFT: derivative landscape
# -----------------------------
cosmo_handles1, cosmo_labels1 = [], []
eft_handles1, eft_labels1 = [], []

for gn, (pnames, labels, colors) in GROUPS.items():
    for pn, lab, col in zip(pnames, labels, colors):
        d = cosmo_derivs_d.get(pn, nuis_ad.get(pn))
        if d is None:
            continue
        nd = np.abs(d) / np.abs(P0_fid)
        if np.max(nd) < 1e-10:
            continue

        h, = ax1.semilogy(k_np, nd, lw=1.3, color=col, label=lab)

        if gn == "Cosmo":
            cosmo_handles1.append(h)
            cosmo_labels1.append(lab)
        else:
            eft_handles1.append(h)
            eft_labels1.append(lab)

ax1.set_xlabel(r"$k$ [$h\,\mathrm{Mpc}^{-1}$]", fontsize=18)
ax1.set_ylabel(r"$|\partial P_0 / \partial \theta_i| / P_0$", fontsize=18)
ax1.set_ylim(5e-12, 5e4)

leg1_cosmo = ax1.legend(
    cosmo_handles1,
    cosmo_labels1,
    frameon=False,
    fontsize=16,
    ncol=1,
    loc="upper left",
    bbox_to_anchor=(0.00, 1.025),
)
ax1.add_artist(leg1_cosmo)

leg1_eft = ax1.legend(
    eft_handles1,
    eft_labels1,
    frameon=False,
    fontsize=16,
    ncol=3,
    loc="lower right",
    bbox_to_anchor=(1.00, -0.025),
)

# -----------------------------
# RIGHT: autodiff vs finite diff
# -----------------------------
cosmo_handles2, cosmo_labels2 = [], []
eft_handles2, eft_labels2 = [], []

# Cosmology
for cp, lab, col in [
    ("fsigma8", r"$f\sigma_8$", "#E24A33"),
    ("Mnu", r"$M_\nu$", "#348ABD"),
    ("Omegam", r"$\Omega_m$", "#988ED5"),
]:
    ad = cosmo_ad[cp]
    fd = cosmo_fd_dict[cp]
    scale = np.maximum(np.abs(ad), np.abs(fd))
    mask = scale > 1e-10 * np.max(scale)

    frac = np.full_like(ad, np.nan)
    frac[mask] = np.abs(ad[mask] - fd[mask]) / scale[mask]

    if np.all(np.isnan(frac)) or np.nanmax(frac) < 1e-15:
        continue

    h, = ax2.semilogy(k_np, frac, lw=1.5, color=col, label=lab, alpha=0.9)
    cosmo_handles2.append(h)
    cosmo_labels2.append(lab)

# EFT / nuisance
for gn, (pnames, labels, colors) in GROUPS.items():
    if gn == "Cosmo":
        continue
    for pn, lab, col in zip(pnames, labels, colors):
        ad, fd = nuis_ad.get(pn), nuis_fd.get(pn)
        if ad is None:
            continue

        scale = np.maximum(np.abs(ad), np.abs(fd))
        mask = scale > 1e-10 * np.max(scale)

        frac = np.full_like(ad, np.nan)
        frac[mask] = np.abs(ad[mask] - fd[mask]) / scale[mask]

        if np.all(np.isnan(frac)) or np.nanmax(frac) < 1e-15:
            continue

        h, = ax2.semilogy(k_np, frac, lw=1.0, color=col, label=lab, alpha=0.8)
        eft_handles2.append(h)
        eft_labels2.append(lab)

ax2.set_xlabel(r"$k$ [$h\,\mathrm{Mpc}^{-1}$]", fontsize=18)
ax2.set_ylabel(r"$|\mathrm{autodiff} - \mathrm{numdiff}| / |\max|$", fontsize=18)
ax2.set_ylim(1e-20, 1e-7)

leg2_cosmo = ax2.legend(
    cosmo_handles2,
    cosmo_labels2,
    frameon=False,
    fontsize=16,
    ncol=1,
    loc="upper right",
    bbox_to_anchor=(1.00, 1.00),
)
ax2.add_artist(leg2_cosmo)

leg2_eft = ax2.legend(
    eft_handles2,
    eft_labels2,
    frameon=False,
    fontsize=16,
    ncol=3,
    loc="lower left",
    bbox_to_anchor=(0.00, -0.025),
)

fig.tight_layout()
fig.savefig("paper/figs/deriv_autodiff_vs_numdiff.png", dpi=200, bbox_inches="tight")
plt.show()

## Appendix Figure A2 --- McDonald-Seljak convergence

Single-tracer vs multi-tracer sigma^2(Pm)/Pm^2 as a function of nbar.
The multi-tracer curve converges to the cosmic-variance-free floor
2/N_modes; the single-tracer plateaus above it due to the b-f degeneracy.

In [ ]:
from numpy.polynomial.legendre import leggauss

bA, bB, f_ms, Pm = 1.0, 2.0, 0.8, 1e3  # Pm ~ P(k=0.1) in (Mpc/h)^3
k_ms, dk_ms, V_ms = 0.1, 0.01, 1e9
Nmodes_ms = k_ms**2 * dk_ms * V_ms / (2 * np.pi**2)
mu_gl, w_gl = leggauss(100)

def _sigma2_st(nbar):
    F = np.zeros((2, 2))
    for mi, wi in zip(mu_gl, w_gl):
        sA = bA + f_ms * mi**2
        PAA = sA**2 * Pm + 1.0 / nbar
        D = np.array([sA**2, 2.0 * sA * Pm])
        F += wi * np.outer(D, D) / (2.0 * PAA**2)
    F *= Nmodes_ms / 2.0
    return 1.0 / (F[0, 0] - F[0, 1]**2 / F[1, 1])

def _sigma2_mt(nbar):
    F = np.zeros((3, 3))
    for mi, wi in zip(mu_gl, w_gl):
        sA = bA + f_ms * mi**2
        sB = bB + f_ms * mi**2
        PAA = sA**2 * Pm + 1.0 / nbar
        PBB = sB**2 * Pm + 1.0 / nbar
        PAB = sA * sB * Pm
        C3 = np.array([
            [2*PAA**2,     2*PAB**2,        2*PAA*PAB],
            [2*PAB**2,     2*PBB**2,        2*PBB*PAB],
            [2*PAA*PAB,    2*PBB*PAB,       PAA*PBB + PAB**2]])
        D = np.array([
            [sA**2,   2*sA*Pm, 0.0],
            [sB**2,   0.0,     2*sB*Pm],
            [sA*sB,   sB*Pm,   sA*Pm]])
        F += wi * D.T @ np.linalg.inv(C3) @ D
    F *= Nmodes_ms / 2.0
    Fb = F[1:, 1:]
    Fc = F[0, 1:]
    return 1.0 / (F[0, 0] - Fc @ np.linalg.solve(Fb, Fc))

def _sigma2_mt_fixed_bB(nbar):
    """Multi-tracer with bB fixed, marginalize over bA only."""
    F = np.zeros((2, 2))
    for mi, wi in zip(mu_gl, w_gl):
        sA = bA + f_ms * mi**2
        sB = bB + f_ms * mi**2
        PAA = sA**2 * Pm + 1.0 / nbar
        PBB = sB**2 * Pm + 1.0 / nbar
        PAB = sA * sB * Pm
        C3 = np.array([
            [2*PAA**2,     2*PAB**2,        2*PAA*PAB],
            [2*PAB**2,     2*PBB**2,        2*PBB*PAB],
            [2*PAA*PAB,    2*PBB*PAB,       PAA*PBB + PAB**2]])
        D = np.array([
            [sA**2,   2*sA*Pm],
            [sB**2,   0.0],
            [sA*sB,   sB*Pm]])
        F += wi * D.T @ np.linalg.inv(C3) @ D
    F *= Nmodes_ms / 2.0
    return 1.0 / (F[0, 0] - F[0, 1]**2 / F[1, 1])

nbars_ms = np.logspace(-4, 3, 200)
s2_st = np.array([_sigma2_st(n) for n in nbars_ms])
s2_mt = np.array([_sigma2_mt(n) for n in nbars_ms])
s2_mt_fixB = np.array([_sigma2_mt_fixed_bB(n) for n in nbars_ms])
cv_floor = 2.0 * Pm**2 / Nmodes_ms
print('A2 MS09 arrays ready: nbars_ms, s2_st, s2_mt, s2_mt_fixB, cv_floor')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(nbars_ms, s2_st / Pm**2, ls="-", color="#4C72B0", lw=2,
          label=r"Single-tracer ($b_A$ marginalized)")
ax.loglog(nbars_ms, s2_mt / Pm**2, ls="-", color="#DD8452", lw=2,
          label=r"Multi-tracer ($b_A, b_B$ marginalized)")
#ax.loglog(nbars_ms, s2_mt_fixB / Pm**2, ls="-.", color="#DD8452", lw=1.5, alpha=0.7,
          #label=r"Multi-tracer ($b_B$ fixed)")
ax.axhline(cv_floor / Pm**2, ls="--", color="k", lw=1.2,
           label=r"CV-free floor $2/N_{\rm modes}$")

# Typical survey nbar range
ax.axvspan(5e-4, 1e-3, color="gray", alpha=0.10, zorder=0)
ax.text(7.8e-4, 5e-4, r"PFS and DESI $\bar{n}$", color="gray",
        fontsize=16, ha="center", rotation=90)

ax.set_xlabel(r"$\bar{n}$ [$(h^{-1}\,\mathrm{Mpc})^{-3}$]")
ax.set_ylabel(r"$\sigma^2(P_m)\,/\,P_m^2$")
ax.set_xlim(1e-4, 1e2)
ax.legend(frameon=False, fontsize=16, loc="upper right")
fig.tight_layout()
fig.savefig("paper/figs/ms09_convergence.png", dpi=200, bbox_inches="tight")
plt.show()

## Appendix Figure A3 --- Fisher information density

Conditional vs marginalized Fisher information density dF_ii/dk for
fσ₈, Mν, Ωm. The gap between the two curves shows how much
information is lost to nuisance marginalization at each k.

In [ ]:
from pfsfog.derivatives import dPell_dtheta_autodiff_all_jit, dPell_d_cosmo_all
from pfsfog.covariance import single_tracer_cov
from pfsfog.fisher_full_area import full_area_fisher_per_zbin
from pfsfog.surveys import desi_elg as make_desi_elg
from pfsfog.eft_params import COSMO_PRIOR_SIGMA
from pfsfog.ps1loop_adapter import make_ps1loop_pkmu_func

desi_fi = make_desi_elg()
ells_fi = (0,2,4); kmax_fi = 0.20
k_fi = np.arange(cfg.kmin, kmax_fi + cfg.dk/2, cfg.dk)
zlo_fi, zhi_fi = 0.8, 1.0
z_fi = 0.9
s8_fi = cosmo_d.sigma8(z_fi); f_fi = float(cosmo_d.f(z_fi))
nbar_fi = desi_fi.nbar_eff(zlo_fi, zhi_fi)
b1_fi = desi_fi.b1_of_z(z_fi); V_fi = desi_fi.volume(zlo_fi, zhi_fi)
pk_fi = cosmo_d.pk_data(z_fi)
fid_fi = desi_elg_fiducials(b1_fi, s8_fi)
par_fi = fisher_to_ps1loop_auto(fid_fi, s8_fi, f_fi, cosmo_d.params["h"], nbar_fi)

nuis_arr_fi = dPell_dtheta_autodiff_all_jit(ps, jnp.array(k_fi), pk_fi, par_fi, s8_fi, ells=ells_fi)  # (N_nuis, N_ell, Nk)
cosmo_fi = dPell_d_cosmo_all(ps, jnp.array(k_fi), pk_fi, cosmo_d, par_fi, z_fi, s8_fi, ells_fi)
pkmu_fi = make_ps1loop_pkmu_func(ps, pk_fi, par_fi)
cov_fi = single_tracer_cov(pkmu_fi, k_fi, nbar_fi, V_fi, cfg.dk, ells_fi)

N_C = len(COSMO_NAMES); N_N = len(NUISANCE_NAMES); Np = N_C + N_N
Nk = len(k_fi); Nell = len(ells_fi)
D_fi = np.zeros((Nk, Nell, Np))
for ic, cn in enumerate(COSMO_NAMES):
    if cn in cosmo_fi:
        for il, ell in enumerate(ells_fi):
            if ell in cosmo_fi[cn]:
                D_fi[:,il,ic] = np.asarray(cosmo_fi[cn][ell])
for il in range(Nell):
    D_fi[:, il, N_C:N_C+N_N] = nuis_arr_fi[:, il, :].T

info_cond = {cn: np.zeros(Nk) for cn in COSMO_NAMES}
info_marg = {cn: np.zeros(Nk) for cn in COSMO_NAMES}

for ik in range(Nk):
    ci = np.linalg.inv(cov_fi[ik])
    Fk = D_fi[ik].T @ ci @ D_fi[ik]
    for ic, cn in enumerate(COSMO_NAMES):
        info_cond[cn][ik] = Fk[ic, ic]

bp_d = broad_priors().prior_fisher_diag()
pf = np.zeros(Np); pf[N_C:] = bp_d
for i, cn in enumerate(COSMO_NAMES):
    pf[i] = 1.0 / COSMO_PRIOR_SIGMA[cn]**2
F_cum = np.diag(pf).copy()
prev = {cn: pf[i] for i, cn in enumerate(COSMO_NAMES)}
for ik in range(Nk):
    ci = np.linalg.inv(cov_fi[ik])
    F_cum += D_fi[ik].T @ ci @ D_fi[ik] * cfg.dk
    try:
        C_cum = np.linalg.inv(F_cum)
        for ic, cn in enumerate(COSMO_NAMES):
            mf = 1.0 / C_cum[ic,ic]
            info_marg[cn][ik] = (mf - prev[cn]) / cfg.dk
            prev[cn] = mf
    except: pass

In [ ]:
clabels_fi = {"fsigma8": r"$f\sigma_8$", "Mnu": r"$M_\nu$", "Omegam": r"$\Omega_m$"}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ic, (cn, ax) in enumerate(zip(COSMO_NAMES, axes)):
    ax.semilogy(k_fi, info_cond[cn], "-", color="#4C72B0", lw=2, label="Conditional")
    ax.semilogy(k_fi, np.maximum(info_marg[cn], 1e-10), "-", color="#DD8452", lw=2, label="Marginalized")
    ax.set_xlabel(r"$k$ [$h\,\mathrm{Mpc}^{-1}$]"); ax.set_title(clabels_fi[cn],fontsize=18)
    if ic == 0: ax.set_ylabel(r"$\mathrm{d}F_{ii}/\mathrm{d}k$", fontsize=16); ax.legend(frameon=False, fontsize=18, loc='upper left', bbox_to_anchor=(0.00,1.03))
fig.tight_layout()
fig.savefig("paper/figs/fisher_info_density.png", dpi=200, bbox_inches="tight")
plt.show()

## Appendix Figure A4 --- Covariance validation (GL vs Wigner 3j)

Validates the 20-point Gauss-Legendre covariance against the analytic
Wigner 3j decomposition (Rubira & Conteddu 2025, Eq. 2.20). Shaded
bands span the range of fractional differences across all covariance
elements (6 spectrum-pair types × 6 multipole combinations). Each step
in ℓ_max gains ~3 orders of magnitude, confirming that the one-loop EFT
model has non-negligible ℓ>4 content and that GL quadrature captures it
automatically.

In [ ]:
from sympy.physics.wigner import wigner_3j as _wigner_3j
from numpy.polynomial.legendre import leggauss as _leggauss
from pfsfog.surveys import pfs_elg as make_pfs_elg, desi_elg as make_desi_elg
from pfsfog.cosmo import FiducialCosmology
from pfsfog.eft_params import tracer_fiducials
from pfsfog.ps1loop_adapter import (
    fisher_to_ps1loop_auto, fisher_to_ps1loop_cross,
    make_ps1loop_pkmu_func, make_ps1loop_pkmu_cross_func,
)
from ps_1loop_jax import PowerSpectrum1Loop

_LS_COV = {
    4: "-",
    6: "--",
    8: "-.",
}
_COL_COV = {
    4: "#4C72B0",
    6: "#7B6EB0",
    8: "#A86CA8",
}

def _legpoly(ell, mu):
    if ell == 0: return np.ones_like(mu)
    if ell == 2: return 0.5 * (3 * mu**2 - 1)
    if ell == 4: return (1/8) * (35 * mu**4 - 30 * mu**2 + 3)
    if ell == 6: return (1/16) * (231 * mu**6 - 315 * mu**4 + 105 * mu**2 - 5)
    if ell == 8: return (1/128) * (6435 * mu**8 - 12012 * mu**6 + 6930 * mu**4 - 1260 * mu**2 + 35)

_w3j_cache = {}
def _w3j_sq(l1, l2, l3):
    key = (l1, l2, l3)
    if key not in _w3j_cache:
        v = float(_wigner_3j(l1, l2, l3, 0, 0, 0))
        _w3j_cache[key] = v * v
    return _w3j_cache[key]

def _decompose(Pkmu, mu, w, ells):
    return {ell: (2*ell+1)/2 * np.sum(_legpoly(ell, mu)[None,:] * Pkmu * w[None,:], axis=1) for ell in ells}

def _cov_w3j(PXW, PYZ, PXZ, PYW, el, elp, Nm, dells):
    res = np.zeros(len(Nm))
    for l1 in dells:
        for l2 in dells:
            for l3 in range(abs(l1-l2), l1+l2+1):
                w1 = _w3j_sq(l1, l2, l3)
                if w1 == 0: continue
                w2 = _w3j_sq(el, elp, l3)
                if w2 == 0: continue
                res += (2*l3+1)*w1*w2*((-1)**elp * PXW[l1]*PYZ[l2] + PXZ[l1]*PYW[l2])
    return (2*el+1)*(2*elp+1)/Nm * res

def _cov_gl(XW, YZ, XZ, YW, el, elp, Nm, mu, w):
    Le, Lep = _legpoly(el, mu), _legpoly(elp, mu)
    integ = Le[None,:]*Lep[None,:]*(XW*YZ + XZ*YW)*w[None,:]
    return (2*el+1)*(2*elp+1)/(2*Nm) * np.sum(integ, axis=1)

# Self-contained setup (no dependency on earlier cells except cfg and np)
cosmo_cv = FiducialCosmology(backend="cosmopower")
ps_cv = PowerSpectrum1Loop(do_irres=False)

pfs_cv = make_pfs_elg(); desi_cv = make_desi_elg()
z_cv = 0.9; s8_cv = cosmo_cv.sigma8(z_cv); f_cv = float(cosmo_cv.f(z_cv)); h_cv = cosmo_cv.params["h"]
pk_cv = cosmo_cv.pk_data(z_cv)
nb_A = pfs_cv.nbar_eff(0.8, 1.0); nb_B = desi_cv.nbar_eff(0.8, 1.0)
b1_A_cv = pfs_cv.b1_of_z(z_cv); b1_B_cv = desi_cv.b1_of_z(z_cv)
fid_Acv = tracer_fiducials("PFS-ELG", b1_A_cv, s8_cv, b1_ref=b1_B_cv, r_sigma_v=cfg.r_sigma_v)
fid_Bcv = tracer_fiducials("DESI-ELG", b1_B_cv, s8_cv)
par_Acv = fisher_to_ps1loop_auto(fid_Acv, s8_cv, f_cv, h_cv, nb_A)
par_Bcv = fisher_to_ps1loop_auto(fid_Bcv, s8_cv, f_cv, h_cv, nb_B)
xpar = fisher_to_ps1loop_cross(fid_Acv, fid_Bcv, s8_cv, f_cv, h_cv, nb_A, nb_B)
pkmu_Acv = make_ps1loop_pkmu_func(ps_cv, pk_cv, par_Acv)
pkmu_Bcv = make_ps1loop_pkmu_func(ps_cv, pk_cv, par_Bcv)
pkmu_ABcv = make_ps1loop_pkmu_cross_func(ps_cv, pk_cv, xpar)

k_cv = np.arange(cfg.kmin, 0.25 + cfg.dk/2, cfg.dk)
mu_cv, w_cv = _leggauss(20)
V_cv = 5.88e8
Nm_cv = k_cv**2 * cfg.dk * V_cv / (2*np.pi**2)
PAA = np.asarray(pkmu_Acv(k_cv, mu_cv)) + 1/nb_A
PBB = np.asarray(pkmu_Bcv(k_cv, mu_cv)) + 1/nb_B
PAB = np.asarray(pkmu_ABcv(k_cv, mu_cv))

# Compute fractional differences for all elements at each ell_max
ell_pairs_cv = [(0,0),(0,2),(0,4),(2,2),(2,4),(4,4)]
spec_cv = [
    (PAA,PAA,PAA,PAA), (PAA,PBB,PAB,PAB), (PAA,PAB,PAB,PAA),
    (PBB,PBB,PBB,PBB), (PAB,PBB,PBB,PAB), (PAB,PAB,PAB,PAB),
]
results_cv = {}
for lmax in [4, 6, 8]:
    dells = tuple(range(0, lmax+1, 2))
    pAA = _decompose(PAA, mu_cv, w_cv, dells)
    pBB = _decompose(PBB, mu_cv, w_cv, dells)
    pAB = _decompose(PAB, mu_cv, w_cv, dells)
    pm = {id(PAA): pAA, id(PBB): pBB, id(PAB): pAB}
    fracs = []
    for (xw, yz, xz, yw) in spec_cv:
        for (el, elp) in ell_pairs_cv:
            cgl = _cov_gl(xw, yz, xz, yw, el, elp, Nm_cv, mu_cv, w_cv)
            cw3 = _cov_w3j(pm[id(xw)], pm[id(yz)], pm[id(xz)], pm[id(yw)], el, elp, Nm_cv, dells)
            with np.errstate(divide="ignore", invalid="ignore"):
                fr = np.abs(cgl - cw3) / np.abs(cgl)
            fr[np.abs(cgl) < 1e-30] = np.nan
            fracs.append(fr)
    results_cv[lmax] = np.array(fracs)

# Plot
print('A4 covariance arrays ready.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for lmax in [4, 6, 8]:
    fr = results_cv[lmax]
    lo = np.nanmin(fr, axis=0); hi = np.nanmax(fr, axis=0)
    med = np.nanmedian(fr, axis=0)
    ax.fill_between(k_cv, lo, hi, color=_COL_COV[lmax], alpha=0.28)
    ax.semilogy(
     k_cv,
     med,
     color=_COL_COV[lmax],
     ls=_LS_COV[lmax],
     lw=1.8,
     label=rf"$\ell_{{\max}}={lmax}$",
)
ax.set_xlabel(r"$k\;[h\,\mathrm{Mpc}^{-1}]$")
ax.set_ylabel(r"$|\mathrm{Cov}_{\mathrm{GL}} - \mathrm{Cov}_{\mathrm{W3}j}| / |\mathrm{Cov}_{\mathrm{GL}}|$")
ax.set_ylim(1e-18, 1)
ax.legend(loc="lower right", title=r"Wigner 3$j$ truncation",
          frameon=False, fancybox=False, edgecolor="0.8",bbox_to_anchor=(1.0,0.0))
fig.tight_layout()
fig.savefig("paper/figs/cov_gl_vs_wigner3j.png", dpi=200, bbox_inches="tight")
plt.show()

for lmax in [4, 6, 8]:
    print(f"  ell_max={lmax}: max={np.nanmax(results_cv[lmax]):.2e}, "
          f"median={np.nanmedian(results_cv[lmax]):.2e}")

## Fisher contour plot — three scenarios

2D Fisher ellipses (1σ and 2σ) for the three scenarios: DESI single-tracer
with broad nuisance priors (legacy reference, no multi-tracer info),
DESI-only joint multi-tracer Fisher across the 14,000 deg² footprint, and
DESI+PFS joint Fisher adding the PFS-ELG sample in the 1,200 deg² overlap.

Three panels show the key degeneracies: b₁σ₈–Mν (dominant in single-tracer),
b₁σ₈–fσ₈, and fσ₈–Mν (marginalized cosmological contour).
b₁σ₈ is shown for the DESI-ELG sample at the representative bin
z=[0.8, 1.0]; cosmology is shared globally across the 8 redshift bins
covering 0.4 < z < 2.1.

In [ ]:
from matplotlib.patches import Ellipse
from pfsfog.config import ForecastConfig
from pfsfog.cosmo import FiducialCosmology
from pfsfog.fisher_joint import run_broad_baseline, run_joint_fisher
from pfsfog.surveys import (
    SurveyGroup, desi_elg, desi_lrg, desi_qso, pfs_elg,
)
from scripts.run_joint_fisher import ZBINS
from ps_1loop_jax import PowerSpectrum1Loop

cfg_fc = ForecastConfig.from_yaml('configs/default.yaml')
cosmo_fc = FiducialCosmology(backend=cfg_fc.cosmo_backend)
ps_fc = PowerSpectrum1Loop(do_irres=False)
sg_fc = SurveyGroup(
    pfs=pfs_elg(),
    desi_tracers={
        'DESI-ELG': desi_elg(),
        'DESI-LRG': desi_lrg(),
        'DESI-QSO': desi_qso(),
    },
    overlap_area_deg2=cfg_fc.overlap_area_deg2,
    desi_full_area_deg2=cfg_fc.desi_area_deg2,
    pfs_zmax=1.6,
)

# `parallel=True` opts into multi-process per-z-bin Fisher (≈1.9× speedup
# with warm JAX cache; cache lives in .cache/jax). Set parallel=False to
# run sequentially (numerically identical, just slower).
PARALLEL = True
N_WORKERS = None  # auto: min(n_zbins, cpu_count, 8)

res_b = run_broad_baseline(cfg_fc, cosmo_fc, ps_fc, sg_fc, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)
res_d = run_joint_fisher(cfg_fc, cosmo_fc, ps_fc, sg_fc, include_pfs=False, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)
res_j = run_joint_fisher(cfg_fc, cosmo_fc, ps_fc, sg_fc, include_pfs=True, zbins=ZBINS,
    parallel=PARALLEL, n_workers=N_WORKERS)

Cb = np.linalg.inv(res_b.fisher.F)
Cd = np.linalg.inv(res_d.fisher.F)
Cj = np.linalg.inv(res_j.fisher.F)
pn_b, pn_d, pn_j = res_b.fisher.param_names, res_d.fisher.param_names, res_j.fisher.param_names

label_b = 'b1_sigma8_DESI-ELG_z0.80-1.00'
xf = float(cosmo_fc.f(0.9)) * cosmo_fc.sigma8(0.9)
xm = 0.06
xb = desi_elg().b1_of_z(0.9) * cosmo_fc.sigma8(0.9)

In [ ]:
def sub(C, names, names_pair):
    i, j = names.index(names_pair[0]), names.index(names_pair[1])
    return np.array([[C[i, i], C[i, j]], [C[j, i], C[j, j]]])

def panel(ax, sub_b, sub_d, sub_j, xc, yc, xl, yl):
    scenarios = [
        (sub_b, '#8C8C8C', 'DESI broad (single-tracer)'),
        (sub_d, '#4C72B0', 'DESI-only (multi-tracer)'),
        (sub_j, '#55A868', 'DESI+PFS (multi-tracer)'),
    ]
    for Cpair, color, label in scenarios:
        vals, vecs = np.linalg.eigh(Cpair)
        vals = np.maximum(vals, 1e-30)
        ang = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
        for ns, al in [(1, 0.36), (2, 0.12)]:
            w = 2 * ns * np.sqrt(vals[0])
            h = 2 * ns * np.sqrt(vals[1])
            e = Ellipse(xy=(xc, yc), width=w, height=h, angle=ang,
                        fill=True, facecolor=color, alpha=al,
                        edgecolor=color, linewidth=1.5)
            ax.add_patch(e)
        ax.plot([], [], color=color, lw=2, label=label)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.autoscale_view()
    x1, x2 = ax.get_xlim(); y1, y2 = ax.get_ylim()
    dx = (x2 - x1) * 0.15; dy = (y2 - y1) * 0.15
    ax.set_xlim(x1 - dx, x2 + dx)
    ax.set_ylim(y1 - dy, y2 + dy)
    ax.plot(xc, yc, '+', color='k', ms=8, mew=1.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panel(axes[0],
      sub(Cb, pn_b, (label_b, 'Mnu')),
      sub(Cd, pn_d, (label_b, 'Mnu')),
      sub(Cj, pn_j, (label_b, 'Mnu')),
      xb, xm, r'$b_1\sigma_8\;(z{\sim}0.9)$', r'$M_\nu$ [eV]')
panel(axes[1],
      sub(Cb, pn_b, (label_b, 'fsigma8')),
      sub(Cd, pn_d, (label_b, 'fsigma8')),
      sub(Cj, pn_j, (label_b, 'fsigma8')),
      xb, xf, r'$b_1\sigma_8\;(z{\sim}0.9)$', r'$f\sigma_8$')
panel(axes[2],
      sub(Cb, pn_b, ('fsigma8', 'Mnu')),
      sub(Cd, pn_d, ('fsigma8', 'Mnu')),
      sub(Cj, pn_j, ('fsigma8', 'Mnu')),
      xf, xm, r'$f\sigma_8$', r'$M_\nu$ [eV]')
axes[0].legend(frameon=False, loc='lower right',bbox_to_anchor=(1.025,-0.025))
fig.tight_layout()
fig.savefig('paper/figs/fisher_contours.png', dpi=200, bbox_inches='tight')
plt.show()